In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import statsmodels.api as sm

ID_COL = "PAT_ID"
T_COL = "T"
Y_COLS = ["Y1", "Y2", "Y3", "Y4", "Y5"]

TEST_SIZE = 0.2
RANDOM_STATE = 37

USE_PROPENSITY_WEIGHTING = True
MAX_INTERACTIONS_AUTO = 30

df = pd.read_excel("DataSet_A.xlsx")
df.columns = [c.strip() for c in df.columns]

if "x13" in df.columns and "X13" not in df.columns:
    df = df.rename(columns={"x13": "X13"})
if "chk_hour" in df.columns and "CHK_HOUR" not in df.columns:
    df = df.rename(columns={"chk_hour": "CHK_HOUR"})

dt_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns.tolist()
for c in dt_cols:
    s = df[c].view("int64").astype("float64")
    s[s < 0] = np.nan
    df[c] = s / 1e9

required = [T_COL] + Y_COLS
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[T_COL] = pd.to_numeric(df[T_COL], errors="coerce")
df = df.dropna(subset=[T_COL] + Y_COLS).copy()
df[T_COL] = df[T_COL].astype(int)
df = df.dropna(subset=Y_COLS).reset_index(drop=True)

CAT_COLS = [
    "X16","X17","X19","X21","X22","X33","X34","X38","X39","X41",
    "X43","X44","X45","X46","X47","X48","X49","X50","X54","X55",
    "X56","X59","X60"
]

NUM_COLS = (
    [f"X{i}" for i in range(1,13)] +
    ["X13","X14","X15","X18","CHK_HOUR","X20"] +
    [f"X{i}" for i in range(23,33)] +
    [f"X{i}" for i in range(35,38)] +
    ["X40","X42","X51","X52","X53","X57","X58","X61","X62"]
)

exclude = set([T_COL] + Y_COLS + ([ID_COL] if ID_COL in df.columns else []))
X_all = [c for c in df.columns if c not in exclude]

cat_cols = [c for c in CAT_COLS if c in X_all]
num_cols = [c for c in NUM_COLS if c in X_all]

other_cols = [c for c in X_all if c not in cat_cols and c not in num_cols]
for c in other_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        num_cols.append(c)
    else:
        cat_cols.append(c)

num_cols = [c for c in num_cols if c in X_all]
cat_cols = [c for c in cat_cols if c in X_all]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop",
)

X_main = preprocess.fit_transform(df[X_all])

feat_names = []
feat_names.extend(num_cols)
if cat_cols:
    ohe = preprocess.named_transformers_["cat"].named_steps["onehot"]
    feat_names.extend(ohe.get_feature_names_out(cat_cols).tolist())

T = df[T_COL].to_numpy().reshape(-1, 1)

def select_interactions(y, X, names, k=30):
    model = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.9, 1.0],
        alphas=np.logspace(-4, 2, 60),
        cv=5,
        random_state=RANDOM_STATE
    )
    model.fit(X, y)
    coef_abs = np.abs(model.coef_)
    order = np.argsort(coef_abs)[::-1]
    top = [names[i] for i in order if coef_abs[i] > 1e-12][:k]
    return top

y_anchor = df[Y_COLS[0]].to_numpy()
interaction_features = select_interactions(y_anchor, X_main, feat_names, k=MAX_INTERACTIONS_AUTO)

feat_map = {f: i for i, f in enumerate(feat_names)}
idx = [feat_map[f] for f in interaction_features if f in feat_map]

X_sel = X_main[:, idx]
TX = X_sel * T
X_full = np.hstack([T, X_main, TX])

def compute_iptw_weights(X, T):
    ps_model = LogisticRegression(max_iter=5000, solver="lbfgs")
    ps_model.fit(X, T.ravel())
    ps = ps_model.predict_proba(X)[:, 1]
    ps = np.clip(ps, 1e-3, 1 - 1e-3)
    p_t = T.mean()
    w = np.where(T.ravel() == 1, p_t / ps, (1 - p_t) / (1 - ps))
    return w

weights = compute_iptw_weights(X_main, T) if USE_PROPENSITY_WEIGHTING else np.ones(len(df))

def eval_regression(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return rmse, r2

def run_outcome(y, y_name):
    Xtr, Xte, ytr, yte = train_test_split(
        X_full, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    enet = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.9, 1.0],
        alphas=np.logspace(-4, 2, 60),
        cv=5,
        random_state=RANDOM_STATE
    )
    enet.fit(Xtr, ytr)

    pred = enet.predict(Xte)
    rmse, r2 = eval_regression(yte, pred)

    print("\n" + "=" * 100)
    print(f"Outcome: {y_name}")
    print(f"ElasticNet: RMSE={rmse:.4f} | R^2={r2:.4f}")
    print(f"Interactions used: {len(idx)} (T×X)")

    inter_start = 1 + X_main.shape[1]
    inter_coefs = enet.coef_[inter_start:]
    top = np.argsort(np.abs(inter_coefs))[::-1][:12]

    print("\nTop effect modifiers (T×X) by |coef| from ElasticNet:")
    for i in top:
        if i < len(interaction_features):
            print(f"  T×{interaction_features[i]:50s}  coef={inter_coefs[i]:+.6f}")

    Xsm = sm.add_constant(X_full, has_constant="add")
    fit = sm.WLS(y, Xsm, weights=weights).fit(cov_type="HC3")

    t_coef = float(fit.params[1])
    t_pval = float(fit.pvalues[1])

    print(f"  coef(T) = {t_coef:+.6f} | p-value = {t_pval:.6g}")

    n = X_main.shape[0]
    ones = np.ones((n, 1))
    zeros = np.zeros((n, 1))

    X_full_t1 = np.hstack([ones, X_main, X_sel * ones])
    X_full_t0 = np.hstack([zeros, X_main, X_sel * zeros])

    y1_all = enet.predict(X_full_t1)
    y0_all = enet.predict(X_full_t0)
    te_all = y1_all - y0_all

    return te_all

print(f"Rows: {len(df):,}")
print(f"Raw X columns: {len(X_all)} | numeric: {len(num_cols)} | categorical: {len(cat_cols)}")
print(f"Expanded features after OHE: {len(feat_names):,}")
print(f"Datetime columns converted to numeric: {len(dt_cols)} -> {dt_cols}")

te_out = pd.DataFrame({ID_COL: df[ID_COL].values} if ID_COL in df.columns else {"row_id": np.arange(len(df))})

for ycol in Y_COLS:
    te = run_outcome(df[ycol].to_numpy(), ycol)
    te_out[f"TE_{ycol}"] = te

te_out.to_csv("patient effects test results.csv", index=False)

print("\nSample of patient-specific treatment effects:")
print(te_out.head())

C:\Users\ty100\AppData\Local\Temp\ipykernel_10756\3123311964.py:34: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  s = df[c].view("int64").astype("float64")
C:\Users\ty100\AppData\Local\Temp\ipykernel_10756\3123311964.py:34: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  s = df[c].view("int64").astype("float64")


Rows: 100,000
Raw X columns: 63 | numeric: 40 | categorical: 23
Expanded features after OHE: 881
Datetime columns converted to numeric: 2 -> ['X14', 'X18']

Outcome: Y1
ElasticNet: RMSE=0.0578 | R^2=0.8636
Interactions used: 30 (T×X)

Top effect modifiers (T×X) by |coef| from ElasticNet:
  T×X5                                                  coef=+0.082965
  T×X4                                                  coef=+0.065287
  T×X3                                                  coef=+0.049927
  T×X2                                                  coef=+0.032590
  T×X1                                                  coef=+0.015781
  T×X36                                                 coef=-0.000757
  T×X27                                                 coef=+0.000709
  T×X30                                                 coef=+0.000252
  T×X37                                                 coef=-0.000163
  T×X10                                                 coef=+0.000095
 

In [ ]:
te_out = te_out.rename(columns={
    "TE_Y1": "TE1",
    "TE_Y2": "TE2",
    "TE_Y3": "TE3",
    "TE_Y4": "TE4",
    "TE_Y5": "TE5",
})

# Save as Excel
te_out.to_excel("DatasetA_patient_specific_TEs.xlsx", index=False)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import statsmodels.api as sm

ID_COL = "PAT_ID"
T_COL = "T"
Y_COLS = ["Y1", "Y2", "Y3", "Y4", "Y5"]

TEST_SIZE = 0.2
RANDOM_STATE = 42

USE_PROPENSITY_WEIGHTING = True
MAX_INTERACTIONS_AUTO = 30    

df = pd.read_excel("DataSet_A.xlsx")
df.columns = [c.strip() for c in df.columns]

# Handle x13 - timing issues
if "x13" in df.columns and "X13" not in df.columns:
    df = df.rename(columns={"x13": "X13"})
if "chk_hour" in df.columns and "CHK_HOUR" not in df.columns:
    df = df.rename(columns={"chk_hour": "CHK_HOUR"})

#handle datetime issues/columns
dt_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns.tolist()
for c in dt_cols:
    s = df[c].view("int64").astype("float64")
    s[s < 0] = np.nan  
    df[c] = s / 1e9

required = [T_COL] + Y_COLS
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# round T to 0 or 1 integer
df[T_COL] = pd.to_numeric(df[T_COL], errors="coerce")
df = df.dropna(subset=[T_COL] + Y_COLS).copy()
df[T_COL] = df[T_COL].astype(int)

# Drop rows with missing out
df = df.dropna(subset=Y_COLS).reset_index(drop=True)

CAT_COLS = [
    "X16","X17","X19","X21","X22","X33","X34","X38","X39","X41",
    "X43","X44","X45","X46","X47","X48","X49","X50","X54","X55",
    "X56","X59","X60"
]

NUM_COLS = (
    [f"X{i}" for i in range(1,13)] +
    ["X13","X14","X15","X18","CHK_HOUR","X20"] +
    [f"X{i}" for i in range(23,33)] +
    [f"X{i}" for i in range(35,38)] +
    ["X40","X42","X51","X52","X53","X57","X58","X61","X62"]
)

exclude = set([T_COL] + Y_COLS + ([ID_COL] if ID_COL in df.columns else []))
X_all = [c for c in df.columns if c not in exclude]

# Make sure columns actually exists
cat_cols = [c for c in CAT_COLS if c in X_all]
num_cols = [c for c in NUM_COLS if c in X_all]

# Everything not in the list gets inferred
other_cols = [c for c in X_all if c not in cat_cols and c not in num_cols]
for c in other_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        num_cols.append(c)
    else:
        cat_cols.append(c)

# Final check
num_cols = [c for c in num_cols if c in X_all]
cat_cols = [c for c in cat_cols if c in X_all]

#preprocess the data 

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop",
)

X_main = preprocess.fit_transform(df[X_all])

# expand the feature names
feat_names = []
feat_names.extend(num_cols)
if cat_cols:
    ohe = preprocess.named_transformers_["cat"].named_steps["onehot"]
    feat_names.extend(ohe.get_feature_names_out(cat_cols).tolist())

T = df[T_COL].to_numpy().reshape(-1, 1)


# evaluate effect modifiers

def select_interactions(y, X, names, k=30):
    model = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.9, 1.0],
        alphas=np.logspace(-4, 2, 60),
        cv=5,
        random_state=RANDOM_STATE
    )
    model.fit(X, y)
    coef_abs = np.abs(model.coef_)
    order = np.argsort(coef_abs)[::-1]
    top = [names[i] for i in order if coef_abs[i] > 1e-12][:k]
    return top

# achor Y
y_anchor = df[Y_COLS[0]].to_numpy()
interaction_features = select_interactions(y_anchor, X_main, feat_names, k=MAX_INTERACTIONS_AUTO)

feat_map = {f: i for i, f in enumerate(feat_names)}
idx = [feat_map[f] for f in interaction_features if f in feat_map]

X_sel = X_main[:, idx]
TX = X_sel * T

# Full design matrix: [T, X, T×X_selected]
X_full = np.hstack([T, X_main, TX])


def compute_iptw_weights(X, T):
    ps_model = LogisticRegression(max_iter=5000, solver="lbfgs")
    ps_model.fit(X, T.ravel())
    ps = ps_model.predict_proba(X)[:, 1]
    ps = np.clip(ps, 1e-3, 1 - 1e-3)

    p_t = T.mean()
    w = np.where(T.ravel() == 1, p_t / ps, (1 - p_t) / (1 - ps))
    return w

weights = compute_iptw_weights(X_main, T) if USE_PROPENSITY_WEIGHTING else np.ones(len(df))

def eval_regression(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return rmse, r2

def run_outcome(y, y_name):
    Xtr, Xte, ytr, yte = train_test_split(
        X_full, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    # ElasticNet for discovery (top modifiers)
    enet = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.9, 1.0],
        alphas=np.logspace(-4, 2, 60),
        cv=5,
        random_state=RANDOM_STATE
    )
    enet.fit(Xtr, ytr)

    pred = enet.predict(Xte)
    rmse, r2 = eval_regression(yte, pred)

    print("\n" + "=" * 100)
    print(f"Outcome: {y_name}")
    print(f"ElasticNet: RMSE={rmse:.4f} | R^2={r2:.4f}")
    print(f"Interactions used: {len(idx)} (T×X)")

    # Show top interaction terms by |coef|
    inter_start = 1 + X_main.shape[1]
    inter_coefs = enet.coef_[inter_start:]
    top = np.argsort(np.abs(inter_coefs))[::-1][:12]

    print("\nTop effect modifiers (T×X) by |coef| from ElasticNet:")
    for i in top:
        if i < len(interaction_features):
            print(f"  T×{interaction_features[i]:50s}  coef={inter_coefs[i]:+.6f}")

    # Weighted OLS for inference-style summary
    Xsm = sm.add_constant(X_full, has_constant="add")
    fit = sm.WLS(y, Xsm, weights=weights).fit(cov_type="HC3")

    t_coef = float(fit.params[1])      # T is column 1 after constant
    t_pval = float(fit.pvalues[1])

    print(f"\nWeighted OLS (IPTW={'ON' if USE_PROPENSITY_WEIGHTING else 'OFF'})")
    print(f"  coef(T) = {t_coef:+.6f} | p-value = {t_pval:.6g}")
    print("  Note: heterogeneity is captured by the interaction terms above.")

print(f"Rows: {len(df):,}")
print(f"Raw X columns: {len(X_all)} | numeric: {len(num_cols)} | categorical: {len(cat_cols)}")
print(f"Expanded features after OHE: {len(feat_names):,}")
print(f"Datetime columns converted to numeric: {len(dt_cols)} -> {dt_cols}")

for ycol in Y_COLS:
    run_outcome(df[ycol].to_numpy(), ycol)

print("\nDONE")


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# ----------------------------
# Load
# ----------------------------
DATA_PATH = "DataSet_A.xlsx"
df = pd.read_excel(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

if "x13" in df.columns and "X13" not in df.columns:
    df = df.rename(columns={"x13": "X13"})
if "chk_hour" in df.columns and "CHK_HOUR" not in df.columns:
    df = df.rename(columns={"chk_hour": "CHK_HOUR"})

Y4_COL = "Y4"
T_COL = "T"
Y_COLS = ["Y1","Y2","Y3","Y4","Y5"]
ID_COL = "PAT_ID"

exclude = set([ID_COL, T_COL] + Y_COLS)
X_cols = [c for c in df.columns if c not in exclude]

df = df.dropna(subset=[Y4_COL]).copy()

y4 = df[Y4_COL].to_numpy()

# ----------------------------
# A) Near-identity checks (numeric only)
# ----------------------------
num_cols = [c for c in X_cols if pd.api.types.is_numeric_dtype(df[c])]

print("\n=== A) Near-identity numeric checks (X ≈ Y4) ===")
hits = []
for c in num_cols:
    x = df[c].to_numpy()
    mask = np.isfinite(x) & np.isfinite(y4)
    if mask.sum() < 100:
        continue
    diff = np.abs(x[mask] - y4[mask])
    frac_close = (diff <= 1e-6).mean()
    if frac_close > 0.999:
        hits.append((c, frac_close))
if hits:
    for c, frac in sorted(hits, key=lambda t: -t[1]):
        print(f"  {c:25s} fraction_equal={frac:.6f}")
else:
    print("  None found.")

# ----------------------------
# B) Simple linear transform check (Y4 ~ aX + b) numeric-only
# ----------------------------
print("\n=== B) Single-column linear predictability (numeric) ===")
lin_hits = []
for c in num_cols:
    x = df[c].to_numpy()
    mask = np.isfinite(x) & np.isfinite(y4)
    if mask.sum() < 100:
        continue
    X = x[mask].reshape(-1, 1)
    y = y4[mask]
    lr = LinearRegression()
    lr.fit(X, y)
    r2 = lr.score(X, y)
    if r2 > 0.9999:
        lin_hits.append((c, r2, float(lr.coef_[0]), float(lr.intercept_)))
if lin_hits:
    for c, r2, a, b in sorted(lin_hits, key=lambda t: -t[1]):
        print(f"  {c:25s} R2={r2:.6f}  Y4≈({a:.6f})*{c}+({b:.6f})")
else:
    print("  None with R2 > 0.9999.")

# ----------------------------
# C) Correlations (numeric only)
# ----------------------------
print("\n=== C) Top |correlations| with Y4 (numeric only) ===")
corrs = []
for c in num_cols:
    x = df[c].to_numpy()
    mask = np.isfinite(x) & np.isfinite(y4)
    if mask.sum() < 100:
        continue
    corr = np.corrcoef(x[mask], y4[mask])[0, 1]
    if np.isfinite(corr):
        corrs.append((c, corr))
for c, corr in sorted(corrs, key=lambda t: abs(t[1]), reverse=True)[:25]:
    print(f"  {c:25s} corr={corr:+.6f}")

# ----------------------------
# D) Train/Test leakage test with SPARSE one-hot
# ----------------------------
cat_cols = [c for c in X_cols if c not in num_cols]

numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True))  # <-- SPARSE!
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ],
    remainder="drop",
    sparse_threshold=0.3  # encourage sparse output
)

X = df[X_cols]
y = df[Y4_COL].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Xtr = preprocess.fit_transform(X_train)
Xte = preprocess.transform(X_test)

# Use SGDRegressor (works well with huge sparse matrices)
model = SGDRegressor(
    loss="squared_error",
    penalty="l2",
    alpha=1e-6,
    max_iter=2000,
    tol=1e-3,
    random_state=42
)
model.fit(Xtr, y_train)

pred = model.predict(Xte)
r2 = r2_score(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("\n=== D) Train/Test predictability check (SGDRegressor on sparse OHE) ===")
print(f"Test R2  = {r2:.6f}")
print(f"Test RMSE= {rmse:.6f}")
if r2 > 0.999:
    print("🚨 Extremely high TEST R2 suggests leakage or near-determinism.")
else:
    print("Test R2 not near 1.0 — less likely classic leakage.")

# ----------------------------
# E) Most influential features (approx)
# ----------------------------
# Getting full one-hot feature names can be huge; instead we report:
# - top numeric coefficients
# - and the largest-magnitude overall coefficients count
print("\n=== E) Coefficient sanity checks ===")
coef = model.coef_
print(f"Total features (after encoding): {coef.shape[0]:,}")
print(f"Coef abs max: {np.max(np.abs(coef)):.6f} | median: {np.median(np.abs(coef)):.6f}")
print("Top numeric predictors by |coef| (numeric part only):")
n_num = len(num_cols)
top_num = np.argsort(np.abs(coef[:n_num]))[::-1][:20]
for i in top_num:
    print(f"  {num_cols[i]:25s} coef={coef[i]:+.6f}")


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error


DATA_PATH = "DataSet_A.xlsx"
df = pd.read_excel(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

if "x13" in df.columns and "X13" not in df.columns:
    df = df.rename(columns={"x13": "X13"})
if "chk_hour" in df.columns and "CHK_HOUR" not in df.columns:
    df = df.rename(columns={"chk_hour": "CHK_HOUR"})

Y4_COL = "Y4"
T_COL = "T"
Y_COLS = ["Y1","Y2","Y3","Y4","Y5"]
ID_COL = "PAT_ID"

exclude = set([ID_COL, T_COL] + Y_COLS)
X_cols = [c for c in df.columns if c not in exclude]

df = df.dropna(subset=[Y4_COL]).copy()
y = df[Y4_COL].astype(float)

# split numeric vs categorical
num_cols = [c for c in X_cols if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in X_cols if c not in num_cols]

# preprocessing: impute + SCALE numeric; one-hot categorical as SPARSE
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))  # key for sparse compatibility
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ],
    remainder="drop",
    sparse_threshold=0.3
)

X = df[X_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Xtr = preprocess.fit_transform(X_train)
Xte = preprocess.transform(X_test)

# Ridge is stable and handles multicollinearity / huge sparse matrices well
ridge = Ridge(alpha=1.0, solver="sag", random_state=42, max_iter=5000)
ridge.fit(Xtr, y_train)

pred = ridge.predict(Xte)

r2 = r2_score(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("\n=== Y4 Predictability Test (Scaled Sparse Ridge) ===")
print(f"Test R2   = {r2:.6f}")
print(f"Test RMSE = {rmse:.6f}")

if r2 > 0.99:
    print("🚨 Y4 is extremely predictable from X → likely leakage or deterministic construction.")
elif r2 > 0.90:
    print("⚠️ Y4 is highly predictable from X → could be engineered from X; inspect top features.")
else:
    print("✅ Y4 is not near-deterministic from X → classic leakage is unlikely.")


In [ ]:
from sklearn.ensemble import RandomForestRegressor

small_cols = [f"X{i}" for i in range(1,13) if f"X{i}" in df.columns]
X_small = df[small_cols].fillna(df[small_cols].median())
y = df["Y4"].astype(float)

Xtr, Xte, ytr, yte = train_test_split(X_small, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
pred = rf.predict(Xte)

print("RF (X1..X12 only) Test R2:", r2_score(yte, pred))


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm   # progress bar

from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

from scipy import sparse


# ============================
# SETTINGS
# ============================
DATA_PATH = "DataSet_A.xlsx"
ID_COL = "PAT_ID"
T_COL = "T"
Y_COLS = ["Y1", "Y2", "Y3", "Y4", "Y5"]

N_SPLITS = 5
RANDOM_STATE = 42
MAX_CAT_CARDINALITY = 200


# ============================
# DICTIONARY GROUPS
# ============================
CAT_COLS_DICT = [
    "X16","X17","X19","X21","X22","X33","X34","X38","X39","X41",
    "X43","X44","X45","X46","X47","X48","X49","X50","X54","X55",
    "X56","X59","X60"
]

NUM_COLS_DICT = (
    [f"X{i}" for i in range(1, 13)] +
    ["X13","X14","X15","X18","CHK_HOUR","X20"] +
    [f"X{i}" for i in range(23, 33)] +
    [f"X{i}" for i in range(35, 38)] +
    ["X40","X42","X51","X52","X53","X57","X58","X61","X62"]
)


# ============================
# LOAD + CLEAN
# ============================
print("Loading data...")
df = pd.read_excel(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

if "x13" in df.columns and "X13" not in df.columns:
    df = df.rename(columns={"x13": "X13"})
if "chk_hour" in df.columns and "CHK_HOUR" not in df.columns:
    df = df.rename(columns={"chk_hour": "CHK_HOUR"})

dt_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns.tolist()
for c in dt_cols:
    s = df[c].view("int64").astype(float)
    s[s < 0] = np.nan
    df[c] = s / 1e9

df[T_COL] = pd.to_numeric(df[T_COL], errors="coerce")
df = df.dropna(subset=[T_COL] + Y_COLS).copy()
df[T_COL] = df[T_COL].astype(int)
df = df.reset_index(drop=True)

print(f"Rows loaded: {len(df):,}")


# ============================
# BUILD X SET
# ============================
exclude = set([T_COL] + Y_COLS + ([ID_COL] if ID_COL in df.columns else []))
X_all = [c for c in df.columns if c not in exclude]

cat_cols = [c for c in CAT_COLS_DICT if c in X_all]
num_cols = [c for c in NUM_COLS_DICT if c in X_all]

other_cols = [c for c in X_all if c not in cat_cols and c not in num_cols]
for c in other_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        num_cols.append(c)
    else:
        cat_cols.append(c)

kept_cat = []
for c in cat_cols:
    if df[c].nunique(dropna=True) <= MAX_CAT_CARDINALITY:
        kept_cat.append(c)
cat_cols = kept_cat

print(f"Numeric X: {len(num_cols)} | Categorical kept: {len(cat_cols)}")


# ============================
# PREPROCESSOR
# ============================
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    sparse_threshold=0.3,
)


# ============================
# CROSS-FIT TE WITH STATUS BAR
# ============================
n = len(df)
te_oof = {y: np.zeros(n) for y in Y_COLS}
yhat_oof = {y: np.zeros(n) for y in Y_COLS}

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

total_steps = N_SPLITS * len(Y_COLS)
pbar = tqdm(total=total_steps, desc="Training CV models", ncols=100)

for fold, (tr_idx, te_idx) in enumerate(kf.split(df), start=1):

    df_tr = df.iloc[tr_idx]
    df_te = df.iloc[te_idx]

    Xtr = preprocess.fit_transform(df_tr[num_cols + cat_cols])
    Xte = preprocess.transform(df_te[num_cols + cat_cols])

    Ttr = df_tr[T_COL].to_numpy().reshape(-1,1)
    Tte = df_te[T_COL].to_numpy().reshape(-1,1)

    Xtr_full = sparse.hstack([Xtr, sparse.csr_matrix(Ttr)], format="csr")
    Xte_full = sparse.hstack([Xte, sparse.csr_matrix(Tte)], format="csr")

    T1 = np.ones((len(te_idx),1))
    T0 = np.zeros((len(te_idx),1))

    Xte_T1 = sparse.hstack([Xte, sparse.csr_matrix(T1)], format="csr")
    Xte_T0 = sparse.hstack([Xte, sparse.csr_matrix(T0)], format="csr")

    for y_name in Y_COLS:

        ytr = df_tr[y_name].to_numpy().astype(float)

        model = Ridge(alpha=1.0, solver="sag", random_state=RANDOM_STATE, max_iter=5000)
        model.fit(Xtr_full, ytr)

        yhat_oof[y_name][te_idx] = model.predict(Xte_full)

        y1 = model.predict(Xte_T1)
        y0 = model.predict(Xte_T0)

        te_oof[y_name][te_idx] = y1 - y0

        pbar.update(1)

pbar.close()


# ============================
# CV PERFORMANCE REPORT
# ============================
print("\nOUT-OF-FOLD PERFORMANCE")

for y_name in Y_COLS:
    y_true = df[y_name].to_numpy()
    y_pred = yhat_oof[y_name]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"{y_name}: RMSE={rmse:.4f} | R2={r2:.4f}")


# ============================
# SAVE PATIENT-SPECIFIC TEs
# ============================
out = pd.DataFrame({
    ID_COL: df[ID_COL] if ID_COL in df.columns else np.arange(n)
})

for i, y_name in enumerate(Y_COLS, start=1):
    out[f"TE{i}"] = te_oof[y_name]

out_path = "DatasetA_patient_specific_TEs_OOF.csv"
out.to_csv(out_path, index=False)

print("\nSaved TE predictions to:", out_path)
print("DONE ✅")


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sys
import time

from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from scipy import sparse

# --- tqdm that works in scripts, notebooks, and buffered consoles ---
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


# ============================
# SETTINGS
# ============================
DATA_PATH = "DataSet_A.xlsx"
ID_COL = "PAT_ID"
T_COL = "T"
Y_COLS = ["Y1", "Y2", "Y3", "Y4", "Y5"]

N_SPLITS = 5
RANDOM_STATE = 42
MAX_CAT_CARDINALITY = 200


CAT_COLS_DICT = [
    "X16","X17","X19","X21","X22","X33","X34","X38","X39","X41",
    "X43","X44","X45","X46","X47","X48","X49","X50","X54","X55",
    "X56","X59","X60"
]

NUM_COLS_DICT = (
    [f"X{i}" for i in range(1, 13)] +
    ["X13","X14","X15","X18","CHK_HOUR","X20"] +
    [f"X{i}" for i in range(23, 33)] +
    [f"X{i}" for i in range(35, 38)] +
    ["X40","X42","X51","X52","X53","X57","X58","X61","X62"]
)


def log(msg: str):
    print(msg, flush=True)


# ============================
# LOAD + CLEAN
# ============================
log("Step 1/6: Loading data...")
df = pd.read_excel(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

if "x13" in df.columns and "X13" not in df.columns:
    df = df.rename(columns={"x13": "X13"})
if "chk_hour" in df.columns and "CHK_HOUR" not in df.columns:
    df = df.rename(columns={"chk_hour": "CHK_HOUR"})

# convert datetime -> numeric seconds
dt_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns.tolist()
for c in dt_cols:
    s = df[c].view("int64").astype(float)
    s[s < 0] = np.nan
    df[c] = s / 1e9

df[T_COL] = pd.to_numeric(df[T_COL], errors="coerce")
df = df.dropna(subset=[T_COL] + Y_COLS).copy()
df[T_COL] = df[T_COL].astype(int)
df = df.reset_index(drop=True)

log(f"Loaded {len(df):,} rows. Datetime converted: {dt_cols}")


# ============================
# BUILD X SET
# ============================
log("Step 2/6: Building feature lists...")
exclude = set([T_COL] + Y_COLS + ([ID_COL] if ID_COL in df.columns else []))
X_all = [c for c in df.columns if c not in exclude]

cat_cols = [c for c in CAT_COLS_DICT if c in X_all]
num_cols = [c for c in NUM_COLS_DICT if c in X_all]

# infer remaining
other_cols = [c for c in X_all if c not in cat_cols and c not in num_cols]
for c in other_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        num_cols.append(c)
    else:
        cat_cols.append(c)

# cardinality cap
kept_cat, dropped = [], []
for c in cat_cols:
    n = df[c].nunique(dropna=True)
    if n <= MAX_CAT_CARDINALITY:
        kept_cat.append(c)
    else:
        dropped.append((c, n))
cat_cols = kept_cat

log(f"Numeric X: {len(num_cols)} | Categorical kept: {len(cat_cols)}")
if dropped:
    log("Dropped high-card categorical columns (first 10 shown):")
    for c, n in dropped[:10]:
        log(f"  - {c}: {n} unique")


# ============================
# PREPROCESSOR
# ============================
log("Step 3/6: Creating preprocess pipeline...")
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    sparse_threshold=0.3,
)

log("Preprocess pipeline ready.")


# ============================
# CROSS-FIT TE WITH PROGRESS
# ============================
log("Step 4/6: Starting cross-fitting (this can take a while)...")

n = len(df)
te_oof = {y: np.zeros(n) for y in Y_COLS}
yhat_oof = {y: np.zeros(n) for y in Y_COLS}

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

total_steps = N_SPLITS * len(Y_COLS)

use_tqdm = tqdm is not None
pbar = tqdm(total=total_steps, desc="CV folds × outcomes", file=sys.stdout) if use_tqdm else None

start = time.time()
done = 0

for fold, (tr_idx, te_idx) in enumerate(kf.split(df), start=1):
    log(f"\nFold {fold}/{N_SPLITS}: fitting preprocess + models...")

    df_tr = df.iloc[tr_idx]
    df_te = df.iloc[te_idx]

    # Fit preprocess on train only
    Xtr = preprocess.fit_transform(df_tr[num_cols + cat_cols])
    Xte = preprocess.transform(df_te[num_cols + cat_cols])

    Ttr = df_tr[T_COL].to_numpy().reshape(-1, 1)
    Tte = df_te[T_COL].to_numpy().reshape(-1, 1)

    Xtr_full = sparse.hstack([Xtr, sparse.csr_matrix(Ttr)], format="csr")
    Xte_full = sparse.hstack([Xte, sparse.csr_matrix(Tte)], format="csr")

    # Counterfactual test matrices
    T1 = np.ones((len(te_idx), 1))
    T0 = np.zeros((len(te_idx), 1))
    Xte_T1 = sparse.hstack([Xte, sparse.csr_matrix(T1)], format="csr")
    Xte_T0 = sparse.hstack([Xte, sparse.csr_matrix(T0)], format="csr")

    for y_name in Y_COLS:
        ytr = df_tr[y_name].to_numpy().astype(float)

        model = Ridge(alpha=1.0, solver="sag", random_state=RANDOM_STATE, max_iter=5000)
        model.fit(Xtr_full, ytr)

        # observed prediction
        yhat_oof[y_name][te_idx] = model.predict(Xte_full)

        # TE = yhat(T=1) - yhat(T=0)
        y1 = model.predict(Xte_T1)
        y0 = model.predict(Xte_T0)
        te_oof[y_name][te_idx] = y1 - y0

        done += 1
        if pbar:
            pbar.update(1)
        else:
            # fallback status line (prints every step)
            elapsed = time.time() - start
            log(f"  completed {done}/{total_steps} steps | last={y_name} | elapsed={elapsed:.1f}s")

if pbar:
    pbar.close()

log("\nStep 5/6: Computing out-of-fold metrics...")


# ============================
# CV PERFORMANCE REPORT
# ============================
for y_name in Y_COLS:
    y_true = df[y_name].to_numpy().astype(float)
    y_pred = yhat_oof[y_name]
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    log(f"{y_name}: RMSE={rmse:.4f} | R2={r2:.4f}")


# ============================
# SAVE PATIENT-SPECIFIC TEs
# ============================
log("Step 6/6: Saving TE outputs...")
out = pd.DataFrame({
    ID_COL: df[ID_COL] if ID_COL in df.columns else np.arange(n)
})

for i, y_name in enumerate(Y_COLS, start=1):
    out[f"TE{i}"] = te_oof[y_name]

out_path = "DatasetA_patient_specific_TEs_OOF.csv"
out.to_csv(out_path, index=False)

log(f"Saved: {out_path}")
log("DONE ✅")


In [ ]:

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# Load and explore the data
# ==============================================================================
print("Loading data...")
df = pd.read_excel("DataSet_A.xlsx")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head(3))
print("\nTreatment distribution:")
print(df['T'].value_counts())
print("\nTotal missing values:", df.isnull().sum().sum())

# ==============================================================================
# Check what we're working with - diagnostics
# ==============================================================================
print("\n" + "="*60)
print("DIAGNOSTICS")
print("="*60)

# Found this annoying inconsistency in the data - one column is lowercase
if 'x13' in df.columns:
    print("\nFound lowercase 'x13' - fixing to match others")
else:
    print("\nColumn names look good")

# Check missing data
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"\nVariables with missing data: {len(missing)}")
print("\nTop 10 variables with most missing:")
print(missing.head(10))

# Make sure outcomes don't have missing values
print("\nOutcome variables (Y1-Y5):")
for outcome in ['Y1', 'Y2', 'Y3', 'Y4', 'Y5']:
    miss_count = df[outcome].isnull().sum()
    miss_pct = (miss_count / len(df)) * 100
    print(f"   {outcome}: {miss_count} missing ({miss_pct:.2f}%)")

# Treatment variable
print("\nTreatment variable:")
print(f"   Distribution: {df['T'].value_counts().to_dict()}")
print(f"   Missing T: {df['T'].isnull().sum()}")

# Figure out which columns are categorical vs numeric
X_cols = [c for c in df.columns if c.startswith('X') or c.startswith('x') or c == 'chk_hour']
print(f"\nFound {len(X_cols)} predictor variables")

categorical_cols = df[X_cols].select_dtypes(include=['object']).columns.tolist()
numeric_cols = [c for c in X_cols if c not in categorical_cols]

print(f"   Categorical: {len(categorical_cols)}")
print(f"   Numeric: {len(numeric_cols)}")

if len(categorical_cols) > 0:
    print(f"\nFirst few categorical columns:")
    for col in categorical_cols[:5]:
        n_unique = df[col].nunique()
        print(f"     {col}: {n_unique} unique values")

# Quick check on outcome ranges
print("\nOutcome variable ranges:")
for outcome in ['Y1', 'Y2', 'Y3', 'Y4', 'Y5']:
    print(f"   {outcome}: min={df[outcome].min():.4f}, max={df[outcome].max():.4f}, mean={df[outcome].mean():.4f}")

# ==============================================================================
# Data preprocessing
# ==============================================================================
print("\n" + "="*60)
print("PREPROCESSING")
print("="*60)

# Fix the lowercase column name if needed
if 'x13' in df.columns:
    df = df.rename(columns={'x13': 'X13'})
    print("\nRenamed x13 -> X13")
    X_cols = [c for c in df.columns if c.startswith('X') or c == 'chk_hour']

# Separate categorical and numeric again after rename
categorical_cols = df[X_cols].select_dtypes(include=['object']).columns.tolist()
numeric_cols = [c for c in X_cols if c not in categorical_cols]

print(f"\nProcessing {len(numeric_cols)} numeric columns and {len(categorical_cols)} categorical columns")

# Handle numeric variables
# Strategy: fill missing with median (more robust to outliers than mean)
X_numeric = df[numeric_cols].copy()
for col in numeric_cols:
    if X_numeric[col].isnull().sum() > 0:
        median_val = X_numeric[col].median()
        X_numeric[col].fillna(median_val, inplace=True)

print(f"Filled missing numeric values with median")

# Handle categorical variables  
# Strategy: fill missing with mode, then label encode
X_categorical = df[categorical_cols].copy()
for col in categorical_cols:
    if X_categorical[col].isnull().sum() > 0:
        mode_val = X_categorical[col].mode()[0]
        X_categorical[col].fillna(mode_val, inplace=True)

print(f"Filled missing categorical values with mode")

# Label encoding for categorical variables
# Need to convert strings to numbers for Random Forest
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_categorical[col] = le.fit_transform(X_categorical[col].astype(str))
    label_encoders[col] = le

print(f"Label encoded {len(categorical_cols)} categorical columns")

# Combine numeric and categorical back together
X_processed = pd.concat([X_numeric, X_categorical], axis=1)
# Reorder to match original column order
X_processed = X_processed[X_cols]

print(f"\nFinal processed feature matrix: {X_processed.shape}")
print(f"Missing values after preprocessing: {X_processed.isnull().sum().sum()}")

# ==============================================================================
# T-Learner Implementation
# ==============================================================================
# The T-Learner approach:
# 1. Split data by treatment group (surgery vs conservative)
# 2. Train separate model for each group
# 3. Predict outcomes for all patients under both treatments
# 4. Treatment effect = prediction(surgery) - prediction(conservative)
#
# Why T-Learner? 
# - Can capture different relationships between features and outcomes for each treatment
# - More flexible than a single model with treatment as a feature
# - Well-suited for this healthcare competition where treatments work differently
# ==============================================================================

print("\n" + "="*60)
print("TRAINING T-LEARNER MODELS")
print("="*60)

# Store all treatment effects
results = pd.DataFrame()
results['PAT_ID'] = df['PAT_ID']

# Track model performance
model_performance = []

# Map outcome codes to what they actually mean
outcome_names = {
    'Y1': 'Long-term pain reduction',
    'Y2': 'Short-term treatment pain',
    'Y3': 'Long-term functional improvement',
    'Y4': 'Rehabilitation time (weeks)',
    'Y5': 'Out-of-pocket costs ($)'
}

# Train models for each of the 5 outcomes
for i, outcome in enumerate(['Y1', 'Y2', 'Y3', 'Y4', 'Y5'], 1):
    print(f"\n{'='*70}")
    print(f"OUTCOME {i}/5: {outcome} - {outcome_names[outcome]}")
    print(f"{'='*70}")
    
    # Get the outcome variable
    y = df[outcome].copy()
    
    # Split data by treatment group
    surgery_mask = df['T'] == 1
    conservative_mask = df['T'] == 0
    
    X_surgery = X_processed[surgery_mask]
    y_surgery = y[surgery_mask]
    
    X_conservative = X_processed[conservative_mask]
    y_conservative = y[conservative_mask]
    
    print(f"   Surgery group: {len(X_surgery):,} patients")
    print(f"   Conservative group: {len(X_conservative):,} patients")
    
    # Train Random Forest for surgery patients
    print("\n   Training surgery model...")
    # Hyperparameters chosen after some experimentation
    # Tried n_estimators=50,100,200 - 100 seemed like good balance
    # max_depth=20 prevents overfitting while capturing complexity
    rf_surgery = RandomForestRegressor(
        n_estimators=100,           
        max_depth=20,               
        min_samples_split=50,       # need at least 50 samples to split
        min_samples_leaf=20,        # leaf nodes need 20+ samples
        max_features='sqrt',        # use sqrt of features at each split
        random_state=42,            
        n_jobs=-1                   # use all cores
    )
    
    rf_surgery.fit(X_surgery, y_surgery)
    
    # Check how well it's doing with cross-validation
    cv_scores_surg = cross_val_score(rf_surgery, X_surgery, y_surgery, 
                                     cv=3, scoring='r2', n_jobs=-1)
    print(f"       Surgery model R²: {cv_scores_surg.mean():.4f} (+/- {cv_scores_surg.std():.4f})")
    
    # Train Random Forest for conservative care patients
    print("\n   Training conservative care model...")
    # Using same hyperparameters for consistency
    rf_conservative = RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_split=50,
        min_samples_leaf=20,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    
    rf_conservative.fit(X_conservative, y_conservative)
    
    cv_scores_cons = cross_val_score(rf_conservative, X_conservative, y_conservative,
                                     cv=3, scoring='r2', n_jobs=-1)
    print(f"       Conservative model R²: {cv_scores_cons.mean():.4f} (+/- {cv_scores_cons.std():.4f})")
    
    # Now predict for ALL patients under BOTH treatments
    print("\n   Calculating treatment effects...")
    
    # What would happen if this patient got surgery?
    pred_surgery = rf_surgery.predict(X_processed)
    # What would happen if this patient got conservative care?
    pred_conservative = rf_conservative.predict(X_processed)
    
    # Treatment effect = difference between the two
    treatment_effect = pred_surgery - pred_conservative
    
    # Save the treatment effects
    results[f'TE{i}'] = treatment_effect
    
    # Look at the distribution of treatment effects
    print(f"\n   Treatment Effect Statistics for {outcome}:")
    print(f"       Mean:   {treatment_effect.mean():>10.4f}")
    print(f"       Std:    {treatment_effect.std():>10.4f}")
    print(f"       Min:    {treatment_effect.min():>10.4f}")
    print(f"       25%:    {np.percentile(treatment_effect, 25):>10.4f}")
    print(f"       Median: {np.percentile(treatment_effect, 50):>10.4f}")
    print(f"       75%:    {np.percentile(treatment_effect, 75):>10.4f}")
    print(f"       Max:    {treatment_effect.max():>10.4f}")
    
    # What features mattered most?
    importance_surg = pd.DataFrame({
        'feature': X_cols,
        'importance': rf_surgery.feature_importances_
    }).sort_values('importance', ascending=False).head(5)
    
    print(f"\n   Top 5 Important Features (Surgery Model):")
    for idx, row in importance_surg.iterrows():
        print(f"       {row['feature']:15s}: {row['importance']:.4f}")
    
    # Keep track of performance metrics
    model_performance.append({
        'Outcome': outcome,
        'Surgery_R2': cv_scores_surg.mean(),
        'Conservative_R2': cv_scores_cons.mean(),
        'TE_Mean': treatment_effect.mean(),
        'TE_Std': treatment_effect.std(),
        'TE_Range': treatment_effect.max() - treatment_effect.min()
    })

# ==============================================================================
# Save results
# ==============================================================================
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

# Load the template they provided
template = pd.read_excel("Treatment Effects.xlsx")

# Fill in our predictions
template[['TE1', 'TE2', 'TE3', 'TE4', 'TE5']] = results[['TE1', 'TE2', 'TE3', 'TE4', 'TE5']]

# Save final submission file
output_file = "Treatment_Effects_FINAL.xlsx"
template.to_excel(output_file, index=False)
print(f"\nResults saved to: {output_file}")

# Also save a CSV backup just in case
csv_file = "Treatment_Effects_FINAL.csv"
template.to_csv(csv_file, index=False)
print(f"Backup CSV saved to: {csv_file}")

# ==============================================================================
# Final summary
# ==============================================================================
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print("\nModel Performance:")
perf_df = pd.DataFrame(model_performance)
print(perf_df.to_string(index=False))

print("\n\nFirst 10 treatment effect predictions:")
print(results.head(10).to_string(index=False))

print("\n" + "="*60)
print("DONE!")
print("="*60)
print(f"\nSubmission file: {output_file}")
print(f"Total patients: {len(results):,}")
print(f"Total predictions: {len(results) * 5:,} (5 outcomes × {len(results):,} patients)")

In [ ]:
# ============================================================
# HDA Competition - Single file (DataSetA) script
# Handles: one dataset file containing both train+test rows
# Strategy: rows with missing targets -> treated as test rows
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error

from sklearn.ensemble import RandomForestRegressor


# -----------------------------
# 1) Load DataSetA
# -----------------------------
DATA_PATH = "DataSetA.csv"   # change to "DataSetA.xlsx" if needed

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Couldn't find {DATA_PATH} in the current directory. "
        "Put it next to this script OR update DATA_PATH."
    )

if DATA_PATH.lower().endswith(".xlsx"):
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print("DataSetA shape:", df.shape)


# -----------------------------
# 2) Identify ID + Targets
# -----------------------------
DEFAULT_TARGETS = ["Y1", "Y2", "Y3", "Y4", "Y5"]
TARGETS = [c for c in DEFAULT_TARGETS if c in df.columns]

if len(TARGETS) == 0:
    raise ValueError(
        "Could not find targets Y1..Y5 in DataSetA. "
        "Edit DEFAULT_TARGETS to match your target columns."
    )

# ID column (common: 'id' or 'ID')
ID_COL = None
for candidate in ["id", "ID", "Id"]:
    if candidate in df.columns:
        ID_COL = candidate
        break

if ID_COL is None:
    ID_COL = "id"
    df[ID_COL] = np.arange(len(df))
    print("⚠️ No id column found; created sequential 'id'.")


# -----------------------------
# 3) Split into train vs test within DataSetA
# -----------------------------
# Primary rule: rows where ANY target is missing are "test"
is_test = df[TARGETS].isna().any(axis=1)
train_df = df.loc[~is_test].copy()
test_df  = df.loc[is_test].copy()

if len(test_df) == 0:
    print("⚠️ No rows with missing targets were found.")
    print("   That means DataSetA might be TRAIN ONLY, or test rows are indicated differently.")
    print("   You can still train + validate, but submission rows cannot be inferred automatically.\n")

print("Train rows:", len(train_df))
print("Test rows :", len(test_df))

X = train_df.drop(columns=TARGETS)
y = train_df[TARGETS]

# If test exists, align features
if len(test_df) > 0:
    X_test = test_df[X.columns].copy()


# -----------------------------
# 4) Preprocessing
# -----------------------------
numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop"
)


# -----------------------------
# 5) Model (Multi-output)
# -----------------------------
base_model = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

model = MultiOutputRegressor(base_model)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model),
])


# -----------------------------
# 6) Quick validation (optional)
# -----------------------------
DO_VALIDATION = True

if DO_VALIDATION:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)

    rmse_per_target = {
        tgt: mean_squared_error(y_val[tgt], val_pred[:, i], squared=False)
        for i, tgt in enumerate(TARGETS)
    }
    avg_rmse = float(np.mean(list(rmse_per_target.values())))

    print("\nValidation RMSE per target:")
    for k, v in rmse_per_target.items():
        print(f"  {k}: {v:,.6f}")
    print(f"Average RMSE: {avg_rmse:,.6f}\n")

# Fit final on all training rows
pipe.fit(X, y)


# -----------------------------
# 7) Predict "test rows" and create submission
# -----------------------------
if len(test_df) > 0:
    test_preds = pipe.predict(X_test)

    submission = pd.DataFrame(test_preds, columns=TARGETS)
    submission.insert(0, ID_COL, test_df[ID_COL].values)

    SUBMISSION_PATH = "submission.csv"
    submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"✅ Saved submission to: {SUBMISSION_PATH}")
    print(submission.head())
else:
    print("No inferred test rows -> no submission.csv written.")
    print("If your dataset uses a flag column (e.g., 'Split'/'Set'/'is_train'), tell me its name and values.")



In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from scipy import sparse


# ============================
# SETTINGS
# ============================
DATA_PATH = "DataSet_A.xlsx"
ID_COL = "PAT_ID"
T_COL = "T"
Y_COLS = ["Y1", "Y2", "Y3", "Y4", "Y5"]

N_SPLITS = 5
RANDOM_STATE = 42
MAX_CAT_CARDINALITY = 200


# ============================
# COLUMN GROUPS
# ============================
CAT_COLS_DICT = [
    "X16","X17","X19","X21","X22","X33","X34","X38","X39","X41",
    "X43","X44","X45","X46","X47","X48","X49","X50","X54","X55",
    "X56","X59","X60"
]

NUM_COLS_DICT = (
    [f"X{i}" for i in range(1,13)] +
    ["X13","X14","X15","X18","CHK_HOUR","X20"] +
    [f"X{i}" for i in range(23,33)] +
    [f"X{i}" for i in range(35,38)] +
    ["X40","X42","X51","X52","X53","X57","X58","X61","X62"]
)


# ============================
# LOAD + CLEAN
# ============================
df = pd.read_excel(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

if "x13" in df.columns and "X13" not in df.columns:
    df = df.rename(columns={"x13": "X13"})
if "chk_hour" in df.columns and "CHK_HOUR" not in df.columns:
    df = df.rename(columns={"chk_hour": "CHK_HOUR"})

dt_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns.tolist()
for c in dt_cols:
    s = df[c].view("int64").astype("float64")
    s[s < 0] = np.nan
    df[c] = s / 1e9

for y in Y_COLS:
    df[y] = pd.to_numeric(df[y], errors="coerce")

df[T_COL] = pd.to_numeric(df[T_COL], errors="coerce")
df = df.dropna(subset=[T_COL] + Y_COLS).copy()
df[T_COL] = df[T_COL].astype(int)
for y in Y_COLS:
    df[y] = df[y].astype(float)

df = df.reset_index(drop=True)
n = len(df)


# ============================
# BUILD X SET
# ============================
exclude = set([T_COL] + Y_COLS + ([ID_COL] if ID_COL in df.columns else []))
X_all = [c for c in df.columns if c not in exclude]

cat_cols = [c for c in CAT_COLS_DICT if c in X_all]
num_cols = [c for c in NUM_COLS_DICT if c in X_all]

other_cols = [c for c in X_all if c not in cat_cols and c not in num_cols]
for c in other_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        num_cols.append(c)
    else:
        cat_cols.append(c)

kept_cat = []
for c in cat_cols:
    if df[c].nunique(dropna=True) <= MAX_CAT_CARDINALITY:
        kept_cat.append(c)
cat_cols = kept_cat


# ============================
# PREPROCESSOR (MATCH SEGMENT 2)
# ============================
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    sparse_threshold=0.3,
)


# ============================
# CROSS-FIT OOF TE (MATCH SEGMENT 2)
# ============================
te_oof = {y: np.zeros(n) for y in Y_COLS}
yhat_oof = {y: np.zeros(n) for y in Y_COLS}

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

total_steps = N_SPLITS * len(Y_COLS)
pbar = tqdm(total=total_steps, desc="Training CV models", ncols=100)

for fold, (tr_idx, te_idx) in enumerate(kf.split(df), start=1):

    df_tr = df.iloc[tr_idx]
    df_te = df.iloc[te_idx]

    Xtr = preprocess.fit_transform(df_tr[num_cols + cat_cols])
    Xte = preprocess.transform(df_te[num_cols + cat_cols])

    Ttr = df_tr[T_COL].to_numpy().reshape(-1, 1)
    Tte = df_te[T_COL].to_numpy().reshape(-1, 1)

    Xtr_full = sparse.hstack([Xtr, sparse.csr_matrix(Ttr)], format="csr")
    Xte_full = sparse.hstack([Xte, sparse.csr_matrix(Tte)], format="csr")

    T1 = np.ones((len(te_idx), 1))
    T0 = np.zeros((len(te_idx), 1))

    Xte_T1 = sparse.hstack([Xte, sparse.csr_matrix(T1)], format="csr")
    Xte_T0 = sparse.hstack([Xte, sparse.csr_matrix(T0)], format="csr")

    for y_name in Y_COLS:
        ytr = df_tr[y_name].to_numpy(dtype=float)

        model = Ridge(alpha=1.0, solver="sag", random_state=RANDOM_STATE, max_iter=5000)
        model.fit(Xtr_full, ytr)

        yhat_oof[y_name][te_idx] = model.predict(Xte_full)

        y1 = model.predict(Xte_T1)
        y0 = model.predict(Xte_T0)

        te_oof[y_name][te_idx] = y1 - y0

        pbar.update(1)

pbar.close()


# ============================
# OOF PERFORMANCE REPORT
# ============================
print("\nOUT-OF-FOLD PERFORMANCE")
for y_name in Y_COLS:
    y_true = df[y_name].to_numpy(dtype=float)
    y_pred = yhat_oof[y_name]

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))

    print(f"{y_name}: RMSE={rmse:.4f} | R2={r2:.4f}")


# ============================
# SAVE PATIENT-SPECIFIC TEs
# ============================
out = pd.DataFrame({
    ID_COL: df[ID_COL] if ID_COL in df.columns else np.arange(n)
})

for i, y_name in enumerate(Y_COLS, start=1):
    out[f"TE{i}"] = te_oof[y_name]

out_path = "DatasetA_patient_specific_TEs_OOF.csv"
out.to_csv(out_path, index=False)

print("\nSaved TE predictions to:", out_path)
print("DONE ✅")


Training CV models: 100%|███████████████████████████████████████████| 25/25 [00:17<00:00,  1.39it/s]



OUT-OF-FOLD PERFORMANCE
Y1: RMSE=0.0742 | R2=0.7764
Y2: RMSE=0.0693 | R2=0.7018
Y3: RMSE=0.0751 | R2=0.7527
Y4: RMSE=2059.4299 | R2=0.9478
Y5: RMSE=1.5066 | R2=0.7383

Saved TE predictions to: DatasetA_patient_specific_TEs_OOF.csv
DONE ✅
